# 1. Fig.3

In [ ]:
library(schard)
library(Seurat)
library(ggplot2)
library(Seurat)
library(monocle3)
library(SeuratWrappers)

In [ ]:
seu_obj = schard::h5ad2seurat('adulte_oocyte.h5ad')
cds <- as.cell_data_set(seu_obj)
reducedDims(cds)$UMAP <- as.matrix(reducedDims(cds)[["XUMAP_"]])

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
cds <- cluster_cells(cds, resolution = 0.08, k = 42)
cds <- learn_graph(cds)

In [ ]:
#use the umap from monoclon3
library(dplyr)
root_cells <- colData(cds) %>%
  as.data.frame() %>%
  filter(leiden_sub_0.6 == "3") %>% 
   rownames() 

if(length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells[1], reduction_method = "UMAP") 
} else {
  message()
}

In [ ]:
custom_palette <- c(
 
    "2"= "#8B45C8",  
    "3"= "#F07A20",  
    "8"= "#3DBD3D" 
)

## 1.1 Fig.3g

In [ ]:
options(repr.plot.width = 5, repr.plot.height =5)
library(patchwork)
library(ggplot2)

p1=plot_cells(cds,
           color_cells_by = "leiden_sub_0.6", 
           label_groups_by_cluster = FALSE,
           label_leaves = FALSE,
           label_branch_points = FALSE,
           graph_label_size = 0, 
           trajectory_graph_segment_size = 3,trajectory_graph_color = "blue",
           cell_size =2)+theme(legend.position = "right")+
          scale_color_manual(values = custom_palette)
ggsave('P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 3/monoclone3/monoclone3_oocyte_cluster_1.pdf', plot = p1, width = 5, height = 5)
p1

In [ ]:

counts_mat <- seu_obj[["RNA"]]$data


cds2 <- new_cell_data_set(
  expression_data = counts_mat,
  cell_metadata = seu_obj@meta.data
)


reducedDims(cds2) <- reducedDims(cds)
cds2 <- estimate_size_factors(cds2)


cds2@principal_graph <- cds@principal_graph
cds2@principal_graph_aux <- cds@principal_graph_aux

In [ ]:
gene_fits <- graph_test(cds2, neighbor_graph = "principal_graph", cores = 12)

In [ ]:
library(dplyr)
sig_genes_strict <- gene_fits %>%
    filter(
        q_value < 0.05,          
        morans_I > 0.30,         
        status == "OK"
    ) %>%
    arrange(desc(morans_I))

nrow(sig_genes_strict)

## 1.1 Fig.3i

In [ ]:
library(ComplexHeatmap)
library(circlize)
options(repr.plot.width =8, repr.plot.height =8)

pt_all <- pseudotime(cds2)
pt_all <- pt_all[is.finite(pt_all)]
cells_ordered <- names(sort(pt_all))


expr_mat <- as.matrix(exprs(cds2[rownames(sig_genes_strict), cells_ordered]))


window_size <- 10
n_windows <- floor(ncol(expr_mat) / window_size)

expr_smooth <- matrix(0, nrow = nrow(expr_mat), ncol = n_windows)
rownames(expr_smooth) <- rownames(expr_mat)

for (i in 1:n_windows) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    expr_smooth[, i] <- rowMeans(expr_mat[, idx])
}


expr_scaled <- t(scale(t(expr_smooth)))
expr_scaled[is.nan(expr_scaled)] <- 0
expr_scaled <- pmin(pmax(expr_scaled, -2), 2)


pt_window <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    mean(pt_all[cells_ordered[idx]], na.rm = TRUE)
})


leiden_colors <- c(
      "2"= "#8B45C8", 
    "3"= "#F07A20",  
    "8"= "#3DBD3D"  
)

leiden_prop <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    cells_win <- cells_ordered[idx]
    tb <- table(factor(colData(cds2)[cells_win, "leiden_sub_0.6"],
                       levels = names(leiden_colors)))
    tb / sum(tb)
})
leiden_prop_mat <- t(leiden_prop)  # n_windows x n_clusters


smooth_peak <- function(x, k = 5) {
    x_smooth <- stats::filter(x, rep(1/k, k), sides = 2)
    x_smooth[is.na(x_smooth)] <- x[is.na(x_smooth)]
    which.max(x_smooth)
}

peak_position <- apply(expr_scaled, 1, smooth_peak)
n_col <- ncol(expr_scaled)
gene_order <- order(peak_position)

set.seed(42)
#km <- kmeans(peak_position, centers = 3,nstart = 25)
#center_order <- order(km$centers)
#levels_map <- setNames(c("Early", "Mid", "Late"), center_order)
#gene_module_ordered <- factor(levels_map[as.character(km$cluster)],
                             #levels = c("Early", "Mid", "Late"))
gene_module_ordered <- cut(peak_position,
                           breaks = c(0, 100, 571, max(peak_position)),
                           labels = c("Early", "Mid", "Late"))

col_fun <- colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))



library(viridis)

viridis_fun <- colorRamp2(
    seq(min(pt_window), max(pt_window), length.out = 4),
    viridis(4)
)




col_anno <- HeatmapAnnotation(
    Pseudotime = anno_simple(
    pt_window,
    col = viridis_fun,
    height = unit(4, "mm")
),
    Leiden_sub = anno_barplot(
        leiden_prop_mat,
        gp = gpar(fill = leiden_colors, col = NA),
        height = unit(8, "mm"),
        bar_width = 1
    ),
    annotation_name_side = "left",
    annotation_name_gp = gpar(fontsize = 9)
)


leiden_legend <- Legend(
    labels = names(leiden_colors),
    legend_gp = gpar(fill = leiden_colors),
    title = "Leiden_sub",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8)
)

pt_legend <- Legend(
    col_fun = viridis_fun,
    title = "Pseudotime",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8),
    direction = "vertical"
)


ht <- Heatmap(
    expr_scaled[gene_order, ],
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = FALSE,
    show_row_names = TRUE,
    row_names_gp = gpar(fontsize = 15),
    col = col_fun,
    row_split = gene_module_ordered[gene_order],
    row_gap = unit(3, "mm"),
    row_title_gp = gpar(fontsize = 15, fontface = "bold"),
    row_title_side = "left",
    border = TRUE,
    row_title_rot = 0,
    column_title = "Smoothed gene expression along pseudotime",
    column_title_gp = gpar(fontsize = 12, fontface = "bold"),
    name = "Z-score",
    top_annotation = col_anno,
    heatmap_legend_param = list(
        title = "Z-score",
        title_gp = gpar(fontsize = 10),
        labels_gp = gpar(fontsize = 8),
        legend_height = unit(4, "cm")
    )
)


pdf("pseudotime_heatmap_oocyte_vivds.pdf", width = 8, height = 8)
draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)
dev.off()

draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)

# 2.Fig. 4

In [ ]:
seu_obj = schard::h5ad2seurat('GC_growing.h5ad')

In [ ]:
cds <- as.cell_data_set(seu_obj)
reducedDims(cds)$UMAP <- as.matrix(reducedDims(cds)[["XUMAP_"]])

options(repr.plot.width = 10, repr.plot.height = 10)
cds <- cluster_cells(cds, resolution = 1e-3, k = 42)
cds <- learn_graph(cds)

In [ ]:
#use the umap from monoclon3
library(dplyr)
root_cells <- colData(cds) %>%
  as.data.frame() %>%
  filter(cluster == "13") %>% 
   rownames()

if(length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells[1], reduction_method = "UMAP") 
} else {
  message()
  
}

In [ ]:
custom_palette <- c(
    "0"  = "#E6194B",
    "1"  = "#3CB44B",
    "2"  = "#FFE119",
    "3"  = "#4363D8",
    "4"  = "#F032E6",#F032E6
    "5"  = "#911EB4",
    "6"  = "#42D4F4",
    "7"  = "#F58231",
    "8"  = "#BFEF45",
    "9"  = "#FABEBE",
    "10" = "#469990",
    "11" = "#E6BEFF",
    "12" = "#9A6324",
    "13" = "#FFFAC8",
    "14" = "#800000",
    "15" = "#AAFFC3",
    "16" = "#808000",
    "17" = "#FFDAC1",
    "18" = "#000075",
    "19" = "#A9A9A9",
    "20" = "#DC143C",
    "21" = "#00FFFF",
    "22" = "#FF1493",
    "23" = "#7FFFD4",
    "24" = "#8B008B"
)

## 2.1 Fig. 4b

In [ ]:
options(repr.plot.width = 5, repr.plot.height =4)
library(patchwork)
library(ggplot2)

p1=plot_cells(cds,
           color_cells_by = "leiden_sub_0.55", 
           label_groups_by_cluster = FALSE,
           label_leaves = FALSE,
           label_branch_points = FALSE,
           graph_label_size = 0, 
           trajectory_graph_segment_size = 1.8,trajectory_graph_color = "blue",
           cell_size = 0.5)+theme(legend.position = "right")+
          scale_color_manual(values = custom_palette)
ggsave('P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 4/monoclone3/monoclone3_growth_GC.png', plot = p1, width = 5, height = 4, dpi = 1200)
p1

In [ ]:

counts_mat <- seu_obj[["RNA"]]$data


cds2 <- new_cell_data_set(
  expression_data = counts_mat,
  cell_metadata = seu_obj@meta.data
)


reducedDims(cds2) <- reducedDims(cds)
cds2 <- estimate_size_factors(cds2)


cds2@principal_graph <- cds@principal_graph
cds2@principal_graph_aux <- cds@principal_graph_aux

In [ ]:
gene_fits <- graph_test(cds2, neighbor_graph = "principal_graph", cores = 12)

In [ ]:
library(dplyr)
sig_genes_strict <- gene_fits %>%
    filter(
        q_value < 0.05,          
        morans_I > 0.25,         
        status == "OK"
    ) %>%
    arrange(desc(morans_I))

nrow(sig_genes_strict)

## 2.2 Fig.4c

In [ ]:
library(ComplexHeatmap)
library(circlize)
options(repr.plot.width =12, repr.plot.height =12)

pt_all <- pseudotime(cds2)
pt_all <- pt_all[is.finite(pt_all)]
cells_ordered <- names(sort(pt_all))


expr_mat <- as.matrix(exprs(cds2[rownames(sig_genes_strict), cells_ordered]))


window_size <- 10
n_windows <- floor(ncol(expr_mat) / window_size)

expr_smooth <- matrix(0, nrow = nrow(expr_mat), ncol = n_windows)
rownames(expr_smooth) <- rownames(expr_mat)

for (i in 1:n_windows) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    expr_smooth[, i] <- rowMeans(expr_mat[, idx])
}


expr_scaled <- t(scale(t(expr_smooth)))
expr_scaled[is.nan(expr_scaled)] <- 0
expr_scaled <- pmin(pmax(expr_scaled, -2), 2)


pt_window <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    mean(pt_all[cells_ordered[idx]], na.rm = TRUE)
})


leiden_colors <- c(
     "0"  = "#E6194B",
    "1"  = "#3CB44B",
    "2"  = "#FFE119",
    "3"  = "#4363D8",
    "4"  = "#F032E6",#F032E6
    "5"  = "#911EB4",
    "6"  = "#42D4F4",
    "7"  = "#F58231",
    "8"  = "#BFEF45",
    "9"  = "#FABEBE",
    "10" = "#469990",
    "11" = "#E6BEFF",
    "12" = "#9A6324",
    "13" = "#FFFAC8",
    "14" = "#800000",
    "15" = "#AAFFC3",
    "16" = "#808000",
    "17" = "#FFDAC1",
    "18" = "#000075",
    "19" = "#A9A9A9",
    "20" = "#DC143C",
    "21" = "#00FFFF",
    "22" = "#FF1493",
    "23" = "#7FFFD4",
    "24" = "#8B008B"
)

leiden_prop <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    cells_win <- cells_ordered[idx]
    tb <- table(factor(colData(cds2)[cells_win, "leiden_sub_0.55"],
                       levels = names(leiden_colors)))
    tb / sum(tb)
})
leiden_prop_mat <- t(leiden_prop)  # n_windows x n_clusters


smooth_peak <- function(x, k = 5) {
    x_smooth <- stats::filter(x, rep(1/k, k), sides = 2)
    x_smooth[is.na(x_smooth)] <- x[is.na(x_smooth)]
    which.max(x_smooth)
}

peak_position <- apply(expr_scaled, 1, smooth_peak)
n_col <- ncol(expr_scaled)
gene_order <- order(peak_position)

set.seed(42)
#km <- kmeans(peak_position, centers = 3,nstart = 25)
#center_order <- order(km$centers)
#levels_map <- setNames(c("Early", "Mid", "Late"), center_order)
#gene_module_ordered <- factor(levels_map[as.character(km$cluster)],
                             #levels = c("Early", "Mid", "Late"))
gene_module_ordered <- cut(peak_position,
                           breaks = c(0, 100, 571, max(peak_position)),
                           labels = c("Early", "Mid", "Late"))

col_fun <- colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))



library(viridis)

viridis_fun <- colorRamp2(
    seq(min(pt_window), max(pt_window), length.out = 4),
    viridis(4)
)




col_anno <- HeatmapAnnotation(
    Pseudotime = anno_simple(
    pt_window,
    col = viridis_fun,
    height = unit(4, "mm")
),
    Leiden_sub = anno_barplot(
        leiden_prop_mat,
        gp = gpar(fill = leiden_colors, col = NA),
        height = unit(8, "mm"),
        bar_width = 1
    ),
    annotation_name_side = "left",
    annotation_name_gp = gpar(fontsize = 9)
)


leiden_legend <- Legend(
    labels = names(leiden_colors),
    legend_gp = gpar(fill = leiden_colors),
    title = "Leiden_sub",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8)
)

pt_legend <- Legend(
    col_fun = viridis_fun,
    title = "Pseudotime",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8),
    direction = "vertical"
)


ht <- Heatmap(
    expr_scaled[gene_order, ],
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = FALSE,
    show_row_names = TRUE,
    row_names_gp = gpar(fontsize = 15),
    col = col_fun,
    row_split = gene_module_ordered[gene_order],
    row_gap = unit(3, "mm"),
    row_title_gp = gpar(fontsize = 15, fontface = "bold"),
    row_title_side = "left",
    border = TRUE,
    row_title_rot = 0,
    column_title = "Smoothed gene expression along pseudotime",
    column_title_gp = gpar(fontsize = 12, fontface = "bold"),
    name = "Z-score",
    top_annotation = col_anno,
    heatmap_legend_param = list(
        title = "Z-score",
        title_gp = gpar(fontsize = 10),
        labels_gp = gpar(fontsize = 8),
        legend_height = unit(4, "cm")
    )
)


#pdf("pseudotime_heatmap_oocyte_vivds.pdf", width = 8, height = 8)
draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)
dev.off()

draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)

## 2.3 Fig 4e

In [ ]:
seu_obj = schard::h5ad2seurat('TC.h5ad')
cds <- as.cell_data_set(seu_obj)
reducedDims(cds)$UMAP <- as.matrix(reducedDims(cds)[["XUMAP_"]])

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
cds <- cluster_cells(cds, resolution = 1e-4, k = 42)
cds <- learn_graph(cds)

In [ ]:
root_cells <- names(clusters(cds))[clusters(cds) == "2"]

if (length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells[1], reduction_method = "UMAP")
} else {
  message()
}

In [ ]:
options(repr.plot.width = 5, repr.plot.height =5)
library(patchwork)
library(ggplot2)

p1=plot_cells(cds,
           color_cells_by = "leiden_sub_0.45",
           label_groups_by_cluster = FALSE,
           label_leaves = FALSE,
           label_branch_points = FALSE,
           graph_label_size = 0, 
           trajectory_graph_segment_size = 1.7,trajectory_graph_color = "blue",
           cell_size = 0.5)+theme(legend.position = "right")+
          scale_color_manual(values = custom_palette)
ggsave('P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 4/monoclone3/monoclone3_atretic_TC.png', plot = p1, width = 5, height = 5, dpi = 1200)
p1

## 2.3 Fig 4f

In [ ]:

counts_mat <- seu_obj[["RNA"]]$data


cds2 <- new_cell_data_set(
  expression_data = counts_mat,
  cell_metadata = seu_obj@meta.data
)


reducedDims(cds2) <- reducedDims(cds)
cds2 <- estimate_size_factors(cds2)


cds2@principal_graph <- cds@principal_graph
cds2@principal_graph_aux <- cds@principal_graph_aux

In [ ]:
custom_palette <- c(
   "0" = "#00FFFF",
    "1" = "#800000",
    "2" = "#AAFFC3",
    "3" = "#808000",
    "4" = "#FFDAC1",
    "5" = "#000075",
    "6" = "#A9A9A9",
    "7" = "#DC143C",
    "8" = "#FFFAC8",
    "9" = "#FF1493",
    "10" = "#7FFFD4",
    "11" = "#8B008B"
)

In [ ]:
gene_fits <- graph_test(cds2, neighbor_graph = "principal_graph", cores = 12)

In [ ]:
library(dplyr)
sig_genes_strict <- gene_fits %>%
    filter(
        q_value < 0.05,          
        morans_I > 0.25,        
        status == "OK"
    ) %>%
    arrange(desc(morans_I))

nrow(sig_genes_strict)

In [ ]:
library(ComplexHeatmap)
library(circlize)
options(repr.plot.width =7, repr.plot.height =5)

pt_all <- pseudotime(cds2)
pt_all <- pt_all[is.finite(pt_all)]
cells_ordered <- names(sort(pt_all))


expr_mat <- as.matrix(exprs(cds2[rownames(sig_genes_strict), cells_ordered]))


window_size <- 10
n_windows <- floor(ncol(expr_mat) / window_size)

expr_smooth <- matrix(0, nrow = nrow(expr_mat), ncol = n_windows)
rownames(expr_smooth) <- rownames(expr_mat)

for (i in 1:n_windows) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    expr_smooth[, i] <- rowMeans(expr_mat[, idx])
}

expr_scaled <- t(scale(t(expr_smooth)))
expr_scaled[is.nan(expr_scaled)] <- 0
expr_scaled <- pmin(pmax(expr_scaled, -2), 2)


pt_window <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    mean(pt_all[cells_ordered[idx]], na.rm = TRUE)
})


leiden_colors <- c(
    "0" = "#00FFFF",
    "1" = "#800000",
    "2" = "#AAFFC3",
    "3" = "#808000",
    "4" = "#FFDAC1",
    "5" = "#000075",
    "6" = "#A9A9A9",
    "7" = "#DC143C",
    "8" = "#FFFAC8",
    "9" = "#FF1493",
    "10" = "#7FFFD4",
    "11" = "#8B008B"
)

leiden_prop <- sapply(1:n_windows, function(i) {
    idx <- ((i-1) * window_size + 1):(i * window_size)
    cells_win <- cells_ordered[idx]
    tb <- table(factor(colData(cds2)[cells_win, "leiden_sub_0.45"],
                       levels = names(leiden_colors)))
    tb / sum(tb)
})
leiden_prop_mat <- t(leiden_prop)  # n_windows x n_clusters


smooth_peak <- function(x, k = 5) {
    x_smooth <- stats::filter(x, rep(1/k, k), sides = 2)
    x_smooth[is.na(x_smooth)] <- x[is.na(x_smooth)]
    which.max(x_smooth)
}

peak_position <- apply(expr_scaled, 1, smooth_peak)
n_col <- ncol(expr_scaled)
gene_order <- order(peak_position)

set.seed(42)
#km <- kmeans(peak_position, centers = 3,nstart = 25)
#center_order <- order(km$centers)
#levels_map <- setNames(c("Early", "Mid", "Late"), center_order)
#gene_module_ordered <- factor(levels_map[as.character(km$cluster)],
                             #levels = c("Early", "Mid", "Late"))
#gene_module_ordered <- cut(peak_position,
                           #breaks = c(0, 100, 571, max(peak_position)),
                           #labels = c("Early", "Mid", "Late"))

col_fun <- colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))



library(viridis)

viridis_fun <- colorRamp2(
    seq(min(pt_window), max(pt_window), length.out = 4),
    viridis(4)
)




col_anno <- HeatmapAnnotation(
    Pseudotime = anno_simple(
    pt_window,
    col = viridis_fun,
    height = unit(4, "mm")
),
    Leiden_sub = anno_barplot(
        leiden_prop_mat,
        gp = gpar(fill = leiden_colors, col = NA),
        height = unit(8, "mm"),
        bar_width = 1
    ),
    annotation_name_side = "left",
    annotation_name_gp = gpar(fontsize = 9)
)


leiden_legend <- Legend(
    labels = names(leiden_colors),
    legend_gp = gpar(fill = leiden_colors),
    title = "Leiden_sub",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8)
)

pt_legend <- Legend(
    col_fun = viridis_fun,
    title = "Pseudotime",
    title_gp = gpar(fontsize = 10, fontface = "bold"),
    labels_gp = gpar(fontsize = 8),
    direction = "vertical"
)


ht <- Heatmap(
    expr_scaled[gene_order, ],
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = FALSE,
    show_row_names = TRUE,
    row_names_gp = gpar(fontsize = 15),
    col = col_fun,
    #row_split = gene_module_ordered[gene_order],
    #row_gap = unit(3, "mm"),
    row_title_gp = gpar(fontsize = 15, fontface = "bold"),
    row_title_side = "left",
    border = TRUE,
    row_title_rot = 0,
    column_title = "Smoothed gene expression along pseudotime",
    column_title_gp = gpar(fontsize = 12, fontface = "bold"),
    name = "Z-score",
    top_annotation = col_anno,
    heatmap_legend_param = list(
        title = "Z-score",
        title_gp = gpar(fontsize = 10),
        labels_gp = gpar(fontsize = 8),
        legend_height = unit(4, "cm")
    )
)


pdf("P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 4/monoclone3/monoclone3_TC_1.pdf", width = 7, height = 5)
draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)
dev.off() 

draw(ht,
     annotation_legend_list = list(leiden_legend, pt_legend),
     merge_legend = TRUE)

# 3. Extended Data Fig. 7  

## 3.1 Extended Data Fig. 7d  

In [ ]:
seu_obj = schard::h5ad2seurat('GC_atretic.h5ad')
cds <- as.cell_data_set(seu_obj)
reducedDims(cds)$UMAP <- as.matrix(reducedDims(cds)[["XUMAP_"]])

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
cds <- cluster_cells(cds, resolution = 1e-3, k = 42)
cds <- learn_graph(cds)

In [ ]:
root_cells <- names(clusters(cds))[clusters(cds) == "1"]

if (length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells[1], reduction_method = "UMAP")
} else {
  message("")
}

In [ ]:
options(repr.plot.width = 6, repr.plot.height =5)
library(patchwork)
library(ggplot2)

p1=plot_cells(cds,
           color_cells_by = "leiden_sub_0.55", 
           label_groups_by_cluster = FALSE,
           label_leaves = FALSE,
           label_branch_points = FALSE,
           graph_label_size = 0, 
           trajectory_graph_segment_size = 2,trajectory_graph_color = "blue",
           cell_size = 0.5)+theme(legend.position = "right")+
          scale_color_manual(values = custom_palette)
#ggsave('P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 4/monoclone3/monoclone3_atretic_GC_1.png', plot = p1, width = 6, height = 5, dpi = 1200)
p1

## 3.2 Extended Data Fig. 7e  

In [ ]:

counts_mat <- seu_obj[["RNA"]]$data


cds2 <- new_cell_data_set(
  expression_data = counts_mat,
  cell_metadata = seu_obj@meta.data
)


reducedDims(cds2) <- reducedDims(cds)
cds2 <- estimate_size_factors(cds2)


cds2@principal_graph <- cds@principal_graph
cds2@principal_graph_aux <- cds@principal_graph_aux

In [ ]:
gene_fits <- graph_test(cds2, neighbor_graph = "principal_graph", cores = 12)

In [ ]:
library(dplyr)
sig_genes_strict <- gene_fits %>%
    filter(
        q_value < 0.05,          
        morans_I > 0.25,        
        status == "OK"
    ) %>%
    arrange(desc(morans_I))

nrow(sig_genes_strict)

In [ ]:
library(ComplexHeatmap)
library(circlize)
options(repr.plot.width = 12, repr.plot.height = 12)

# ===== Step 1: Order all cells by pseudotime =====
pt_all <- pseudotime(cds2)
pt_all <- pt_all[is.finite(pt_all)]
cells_ordered <- names(sort(pt_all))

# ===== Step 2: Extract expression matrix using strictly filtered genes =====
expr_mat <- as.matrix(
    exprs(cds2[rownames(sig_genes_strict), cells_ordered])
)

# ===== Step 3: Smooth expression using a sliding window =====
window_size <- 10
n_windows <- floor(ncol(expr_mat) / window_size)

expr_smooth <- matrix(
    0,
    nrow = nrow(expr_mat),
    ncol = n_windows
)

rownames(expr_smooth) <- rownames(expr_mat)

for (i in 1:n_windows) {
    idx <- ((i - 1) * window_size + 1):(i * window_size)
    expr_smooth[, i] <- rowMeans(expr_mat[, idx])
}

# ===== Step 4: Row-wise scaling =====
expr_scaled <- t(scale(t(expr_smooth)))
expr_scaled[is.nan(expr_scaled)] <- 0
expr_scaled <- pmin(pmax(expr_scaled, -2), 2)

# ===== Step 5: Calculate mean pseudotime and Leiden cluster proportions for each window =====
pt_window <- sapply(1:n_windows, function(i) {
    idx <- ((i - 1) * window_size + 1):(i * window_size)
    mean(
        pt_all[cells_ordered[idx]],
        na.rm = TRUE
    )
})

leiden_colors <- c(
    "0"  = "#E6194B",
    "1"  = "#3CB44B",
    "2"  = "#FFE119",
    "3"  = "#4363D8",
    "4"  = "#F032E6",
    "5"  = "#911EB4",
    "6"  = "#42D4F4",
    "7"  = "#F58231",
    "8"  = "#BFEF45",
    "9"  = "#FABEBE",
    "10" = "#469990",
    "11" = "#E6BEFF",
    "12" = "#9A6324",
    "13" = "#FFFAC8",
    "14" = "#800000",
    "15" = "#AAFFC3",
    "16" = "#808000",
    "17" = "#FFDAC1",
    "18" = "#000075",
    "19" = "#A9A9A9",
    "20" = "#DC143C",
    "21" = "#00FFFF",
    "22" = "#FF1493",
    "23" = "#7FFFD4",
    "24" = "#8B008B"
)

leiden_prop <- sapply(1:n_windows, function(i) {
    idx <- ((i - 1) * window_size + 1):(i * window_size)
    cells_win <- cells_ordered[idx]

    tb <- table(
        factor(
            colData(cds2)[cells_win, "leiden_sub_0.55"],
            levels = names(leiden_colors)
        )
    )

    tb / sum(tb)
})

leiden_prop_mat <- t(leiden_prop)

# ===== Step 6: Order genes by smoothed peak position =====
smooth_peak <- function(x, k = 5) {
    x_smooth <- stats::filter(
        x,
        rep(1 / k, k),
        sides = 2
    )

    x_smooth[is.na(x_smooth)] <- x[is.na(x_smooth)]

    which.max(x_smooth)
}

peak_position <- apply(
    expr_scaled,
    1,
    smooth_peak
)

n_col <- ncol(expr_scaled)

gene_order <- order(
    peak_position
)

set.seed(42)

# km <- kmeans(
#     peak_position,
#     centers = 3,
#     nstart = 25
# )

# center_order <- order(km$centers)

# levels_map <- setNames(
#     c("Early", "Mid", "Late"),
#     center_order
# )

# gene_module_ordered <- factor(
#     levels_map[as.character(km$cluster)],
#     levels = c("Early", "Mid", "Late")
# )

# gene_module_ordered <- cut(
#     peak_position,
#     breaks = c(0, 100, 571, max(peak_position)),
#     labels = c("Early", "Mid", "Late")
# )

# ===== Step 7: Define color scales =====
col_fun <- colorRamp2(
    c(-2, 0, 2),
    c("#2166AC", "white", "#B2182B")
)

library(viridis)

viridis_fun <- colorRamp2(
    seq(
        min(pt_window),
        max(pt_window),
        length.out = 4
    ),
    viridis(4)
)

# ===== Step 8: Column annotations =====
col_anno <- HeatmapAnnotation(
    Pseudotime = anno_simple(
        pt_window,
        col = viridis_fun,
        height = unit(4, "mm")
    ),

    Leiden_sub = anno_barplot(
        leiden_prop_mat,
        gp = gpar(
            fill = leiden_colors,
            col = NA
        ),
        height = unit(8, "mm"),
        bar_width = 1
    ),

    annotation_name_side = "left",
    annotation_name_gp = gpar(
        fontsize = 9
    )
)

# ===== Step 9: Legends =====
leiden_legend <- Legend(
    labels = names(leiden_colors),
    legend_gp = gpar(
        fill = leiden_colors
    ),
    title = "Leiden_sub",
    title_gp = gpar(
        fontsize = 10,
        fontface = "bold"
    ),
    labels_gp = gpar(
        fontsize = 8
    )
)

pt_legend <- Legend(
    col_fun = viridis_fun,
    title = "Pseudotime",
    title_gp = gpar(
        fontsize = 10,
        fontface = "bold"
    ),
    labels_gp = gpar(
        fontsize = 8
    ),
    direction = "vertical"
)

# ===== Step 10: Draw heatmap =====
ht <- Heatmap(
    expr_scaled[gene_order, ],
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    show_column_names = FALSE,
    show_row_names = TRUE,
    row_names_gp = gpar(
        fontsize = 15
    ),
    col = col_fun,
    # row_split = gene_module_ordered[gene_order],
    # row_gap = unit(3, "mm"),
    row_title_gp = gpar(
        fontsize = 15,
        fontface = "bold"
    ),
    row_title_side = "left",
    border = TRUE,
    row_title_rot = 0,
    column_title = "Smoothed gene expression along pseudotime",
    column_title_gp = gpar(
        fontsize = 12,
        fontface = "bold"
    ),
    name = "Z-score",
    top_annotation = col_anno,
    heatmap_legend_param = list(
        title = "Z-score",
        title_gp = gpar(
            fontsize = 10
        ),
        labels_gp = gpar(
            fontsize = 8
        ),
        legend_height = unit(
            4,
            "cm"
        )
    )
)

# ===== Step 11: Save figure =====
pdf(
    "P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/19_Xenium 5K_prepurvty ovary/paper/figure/1st_2026_03_19/Figure 4/monoclone3/monoclone3_atretic_GC.pdf",
    width = 10,
    height = 10
)

draw(
    ht,
    annotation_legend_list = list(
        leiden_legend,
        pt_legend
    ),
    merge_legend = TRUE
)

dev.off()

draw(
    ht,
    annotation_legend_list = list(
        leiden_legend,
        pt_legend
    ),
    merge_legend = TRUE
)